# Yuki VTuber - Servidor de Voz (GPT-SoVITS)

**Execute as celulas em ordem.**
- Celula 1: Verifica GPU
- Celula 2: Instala GPT-SoVITS
- Celula 3: Baixa modelos
- Celula 4: Upload do audio de referencia
- Celula 5: Inicia servidor
- Celula 6: Cria tunel publico (pega a URL aqui)
- Celula 7: Mantem sessao ativa

In [ ]:
# Celula 1 - Verificar GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ERRO: sem GPU - ative em Ambiente de execucao > Alterar tipo')

In [ ]:
# Celula 2 - Instalar GPT-SoVITS + dependencias
import os

if not os.path.exists('/content/GPT-SoVITS'):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS /content/GPT-SoVITS

os.chdir('/content/GPT-SoVITS')
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn ffmpeg-python pytorch_lightning x_transformers fast_langdetect split_lang cn2an jieba_fast
print('\nInstalacao concluida!')

In [ ]:
# Celula 3 - Baixar modelos pre-treinados
import os

LOCAL_MODELS = '/content/GPT-SoVITS/GPT_SoVITS/pretrained_models'
os.makedirs(f'{LOCAL_MODELS}/gsv-v2final-pretrained', exist_ok=True)

# Modelos SoVITS + GPT
if not os.path.exists(f'{LOCAL_MODELS}/gsv-v2final-pretrained/s2G2333k.pth'):
    !wget -q --show-progress -O {LOCAL_MODELS}/gsv-v2final-pretrained/s2G2333k.pth https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s2G2333k.pth
    print('s2G2333k.pth baixado!')
else:
    print('s2G2333k.pth ja existe')

if not os.path.exists(f'{LOCAL_MODELS}/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt'):
    !wget -q --show-progress -O "{LOCAL_MODELS}/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt" "https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch%3D12-step%3D369668.ckpt"
    print('GPT model baixado!')
else:
    print('GPT model ja existe')

# HuBERT e BERT
for m in ['chinese-roberta-wwm-ext-large', 'chinese-hubert-base']:
    if not os.path.exists(f'{LOCAL_MODELS}/{m}'):
        print(f'Clonando {m}...')
        !git clone --depth 1 https://huggingface.co/hfl/{m} {LOCAL_MODELS}/{m}
    else:
        print(f'{m} ja existe')

print('\nModelos prontos!')

In [ ]:
# Celula 4 - Upload do audio de referencia
import os
from google.colab import files

REF_DIR = '/content/ref_audio'
os.makedirs(REF_DIR, exist_ok=True)

existing = [f for f in os.listdir(REF_DIR) if f.endswith('.wav')] if os.path.exists(REF_DIR) else []

if existing:
    print(f'Audio ja existe: {existing[0]}')
else:
    print('Faca upload do arquivo .wav de referencia')
    uploaded = files.upload()
    for fn, data in uploaded.items():
        dest = f'{REF_DIR}/{fn}'
        open(dest, 'wb').write(data)
        print(f'Salvo: {dest}')

In [ ]:
# Celula 5 - Iniciar servidor GPT-SoVITS
import os, subprocess, time, glob
os.chdir('/content/GPT-SoVITS')

# Achar o audio de referencia
ref_files = glob.glob('/content/ref_audio/*.wav')
if not ref_files:
    print('ERRO: Nenhum audio de referencia encontrado! Rode a Celula 4 primeiro.')
else:
    ref = ref_files[0]
    print(f'Audio de referencia: {ref}')

    os.makedirs('GPT_SoVITS/configs', exist_ok=True)
    config = f"""default:
  device: cuda
  is_half: true
  version: v2
  t2s_weights_path: GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt
  vits_weights_path: GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s2G2333k.pth
  bert_base_path: GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large
  cnhuhbert_base_path: GPT_SoVITS/pretrained_models/chinese-hubert-base
  ref_audio_path: {ref}
  prompt_text: Vamos la menino segura essa espada
  prompt_lang: en
  text_lang: en
  text_split_method: cut5
  batch_size: 1
  media_type: wav
  streaming_mode: false
"""
    open('GPT_SoVITS/configs/tts_infer.yaml', 'w').write(config)

    proc = subprocess.Popen(
        ['python', 'api_v2.py', '-a', '0.0.0.0', '-p', '9880', '-c', 'GPT_SoVITS/configs/tts_infer.yaml'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print('Iniciando servidor GPT-SoVITS...')
    for i in range(120):
        l = proc.stdout.readline()
        if l.strip(): print(l.strip())
        if 'startup complete' in l.lower() or 'uvicorn running' in l.lower():
            print('\nServidor ativo na porta 9880!')
            break
        time.sleep(0.5)

In [ ]:
# Celula 6 - Criar tunel publico (COPIE A URL DAQUI)
import subprocess, time, re, os
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
tp = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:9880'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Aguardando URL do tunel...')
for i in range(40):
    l = tp.stdout.readline()
    if 'trycloudflare.com' in l:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print(f'\n=== SERVIDOR ATIVO ===')
            print(f'URL: {url}')
            print(f'\nCole no conf.yaml local do projeto:')
            print(f'  gpt_sovits_tts:')
            print(f'    api_url: "{url}/tts"')
            break
    time.sleep(1)

In [ ]:
# Celula 7 - Manter sessao ativa (nao feche!)
import time
print('Sessao ativa - nao feche esta celula')
print('Enquanto isso, cole a URL do tunel no conf.yaml e inicie o projeto local')
while True:
    time.sleep(60)
    print(f'Online {time.strftime("%H:%M:%S")}')